# 12 - Strict probes, BatchNorm recalibration, and sparse-head control

Makes the leakage-safe probe primary, reruns pairwise probes under the same boundary, isolates stale BatchNorm statistics, and compares dense versus equally sparse decision-layer recovery.

**Safety:** this notebook writes only new files under `results/tables/comnet/` and does not overwrite archived manuscript result tables.

In [ ]:
# Colab/bootstrap cell: no tokens or credentials are required.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time
from pathlib import Path

REPO = Path('/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression')
if not REPO.exists():
    # Local/Jupyter fallback: run the notebook from the repository root.
    REPO = Path.cwd()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.config import CFG, PATHS, set_all_seeds
set_all_seeds(CFG['anchor_seed'])

OUT_TABLE = PATHS.tables('comnet')
OUT_TABLE.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs:', OUT_TABLE)


## Configuration

In [ ]:
DATASET = 'ciciot2023'
ARCH = 'cnn1d'
ARCH_KW = {'channels': (64, 128)}
SEED = int(CFG['anchor_seed'])
RUN_BATCHNORM_CONTROL = True
RUN_SPARSE_HEAD_CONTROL = True
REBUILD_PRUNE80_IF_MISSING = False
FIT_MAX_PER_CLASS = 5000
TEST_MAX_PER_CLASS = 3000
PROBE_BOOTSTRAP_B = 500
BN_MAX_PER_CLASS = 10000
HEAD_TRAIN_MAX_PER_CLASS = None  # set e.g. 20000 only if memory is constrained


In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor, predict, per_class_recall_table, tempered_class_weights
from src import explain, compression
from src.comnet_audit import (
    strict_ovr_probe_table, strict_pairwise_probe_table,
    recalibrate_batchnorm_from_dataframe, calibration_summary,
    stratified_cap_indices, security_semantics,
)

df = clean(load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = temporal_within_capture_split(df, SEED)
m0, le, scaler, feat_cols = load_anchor(DATASET, ARCH, 'M0', SEED, arch_kwargs=ARCH_KW)
try:
    p80, _, _, _ = load_anchor(DATASET, ARCH, 'prune80', SEED, arch_kwargs=ARCH_KW)
except Exception as exc:
    if not REBUILD_PRUNE80_IF_MISSING:
        raise RuntimeError('prune80 checkpoint missing; set REBUILD_PRUNE80_IF_MISSING=True to rebuild') from exc
    p80, _, _ = compression.prune_and_finetune(
        m0, df, DATASET, splits, SEED, 0.80,
        ft_epochs=8, batch_size=4096, lr=5e-4, arch=ARCH, verbose=True,
    )


## Strict train+validation -> untouched test probes

In [ ]:
fit_raw = np.concatenate([splits['train'], splits['val']])
fit_y_raw = le.transform(df.loc[fit_raw, 'label'].to_numpy())
fit_pos = stratified_cap_indices(fit_y_raw, max_per_class=FIT_MAX_PER_CLASS, seed=SEED)
fit_idx = fit_raw[fit_pos]

test_raw = np.asarray(splits['test'])
test_y_raw = le.transform(df.loc[test_raw, 'label'].to_numpy())
test_pos = stratified_cap_indices(test_y_raw, max_per_class=TEST_MAX_PER_CLASS, seed=SEED + 1)
test_idx = test_raw[test_pos]
probe_splits = {'fit': fit_idx, 'probe_test': test_idx}


def extract_probe(model):
    ffit, _, yfit = explain.extract_features(model, df, probe_splits, scaler, feat_cols, le, which='fit')
    ftest, _, ytest = explain.extract_features(model, df, probe_splits, scaler, feat_cols, le, which='probe_test')
    return ffit, yfit, ftest, ytest

fit0, yfit, test0, ytest = extract_probe(m0)
fit8, yfit8, test8, ytest8 = extract_probe(p80)
assert np.array_equal(yfit, yfit8) and np.array_equal(ytest, ytest8)

primary_classes = [
    'DoS-UDP_Flood','DoS-HTTP_Flood','Recon-HostDiscovery','BenignTraffic',
    'Recon-PortScan','DoS-SYN_Flood','DNS_Spoofing','MITM-ArpSpoofing',
    'Recon-OSScan','DictionaryBruteForce','VulnerabilityScan',
    'DDoS-HTTP_Flood','DDoS-UDP_Flood',
]
strict_ovr = strict_ovr_probe_table(
    fit0, test0, fit8, test8, yfit, ytest, le.classes_,
    seed=SEED, bootstrap_B=PROBE_BOOTSTRAP_B, class_filter=primary_classes,
)
strict_ovr.to_csv(OUT_TABLE / 'strict_probe_trainval_to_test.csv', index=False)
display(strict_ovr.sort_values('auc_drop', ascending=False))

pairs = [
    ('DoS-UDP_Flood','DDoS-UDP_Flood'),
    ('DoS-SYN_Flood','DDoS-SYN_Flood'),
    ('DoS-HTTP_Flood','DDoS-HTTP_Flood'),
    ('Recon-PortScan','VulnerabilityScan'),
    ('Recon-OSScan','VulnerabilityScan'),
    ('Recon-HostDiscovery','VulnerabilityScan'),
]
strict_pair = strict_pairwise_probe_table(
    fit0, test0, fit8, test8, yfit, ytest, le.classes_, pairs,
    seed=SEED, bootstrap_B=PROBE_BOOTSTRAP_B,
)
strict_pair.to_csv(OUT_TABLE / 'strict_pairwise_probe_trainval_to_test.csv', index=False)
display(strict_pair)
print('Probe fit/test rows:', len(yfit), len(ytest))


## BatchNorm-only recalibration control

In [ ]:
if RUN_BATCHNORM_CONTROL:
    oneshot = compression._magnitude_prune(m0, 0.80)
    train_raw = np.asarray(splits['train'])
    train_y_raw = le.transform(df.loc[train_raw, 'label'].to_numpy())
    bn_pos = stratified_cap_indices(train_y_raw, max_per_class=BN_MAX_PER_CLASS, seed=SEED + 2)
    bn_idx = train_raw[bn_pos]
    bn_recal = recalibrate_batchnorm_from_dataframe(
        oneshot, df, bn_idx, scaler, feat_cols, batch_size=8192, reset=True
    )

    rows = []
    for name, model in [('M0',m0), ('prune80_oneshot',oneshot), ('prune80_bn_recal',bn_recal)]:
        yt, yp, probs = predict(model, df, splits, le, scaler, feat_cols, which='test')
        cal = calibration_summary(probs,yt).iloc[0].to_dict()
        sec, _, _ = security_semantics(yt, yp, le.classes_)
        rows.append({'condition':name, **cal,
                     'attack_to_benign_rate':sec.loc[0,'attack_to_benign_rate'],
                     'benign_to_attack_rate':sec.loc[0,'benign_to_attack_rate']})
        per_class_recall_table(yt,yp,le).to_csv(OUT_TABLE/f'bn_control_{name}_per_class.csv',index=False)
    bn_summary = pd.DataFrame(rows)
    bn_summary.to_csv(OUT_TABLE/'batchnorm_recalibration_summary.csv',index=False)
    display(bn_summary.round(4))


## Dense versus 80%-sparse replacement head

In [ ]:
if RUN_SPARSE_HEAD_CONTROL:
    train_raw = np.asarray(splits['train'])
    if HEAD_TRAIN_MAX_PER_CLASS is not None:
        y_raw = le.transform(df.loc[train_raw, 'label'].to_numpy())
        pos = stratified_cap_indices(y_raw, max_per_class=HEAD_TRAIN_MAX_PER_CLASS, seed=SEED + 3)
        head_train_idx = train_raw[pos]
    else:
        head_train_idx = train_raw
    head_splits = {'head_train':head_train_idx, 'head_val':splits['val'], 'head_test':splits['test']}
    ftr, _, ytr = explain.extract_features(p80, df, head_splits, scaler, feat_cols, le, which='head_train')
    fva, _, yva = explain.extract_features(p80, df, head_splits, scaler, feat_cols, le, which='head_val')
    fte, _, yte = explain.extract_features(p80, df, head_splits, scaler, feat_cols, le, which='head_test')
    K, d = len(le.classes_), ftr.shape[1]
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'

    def train_head(head, *, mask=None, epochs=20, lr=1e-2, patience=4):
        head = head.to(dev)
        if mask is not None:
            mask = mask.to(dev)
            hook = head.weight.register_hook(lambda g: g * mask)
        else:
            hook = None
        w = tempered_class_weights(ytr,K).to(dev)
        crit = nn.CrossEntropyLoss(weight=w)
        opt = torch.optim.Adam(head.parameters(),lr=lr)
        loader = DataLoader(TensorDataset(torch.tensor(ftr,dtype=torch.float32),
                                          torch.tensor(ytr,dtype=torch.long)),
                            batch_size=4096,shuffle=True)
        Xv = torch.tensor(fva,dtype=torch.float32,device=dev)
        best, best_state, stale = -np.inf, None, 0
        for _ in range(epochs):
            head.train()
            for xb,yb in loader:
                xb,yb=xb.to(dev),yb.to(dev)
                opt.zero_grad(); crit(head(xb),yb).backward(); opt.step()
                if mask is not None:
                    with torch.no_grad(): head.weight.mul_(mask)
            head.eval()
            with torch.no_grad(): vp=head(Xv).argmax(1).cpu().numpy()
            score=f1_score(yva,vp,average='macro')
            if score > best + 1e-4:
                best=score; best_state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}; stale=0
            else:
                stale += 1
                if stale >= patience: break
        if hook is not None: hook.remove()
        head.load_state_dict(best_state); head.eval()
        with torch.no_grad(): L=head(torch.tensor(fte,dtype=torch.float32,device=dev)).cpu().numpy()
        return head.cpu(), L, best

    dense_init = nn.Linear(d,K)
    dense_head, dense_logits, dense_val_f1 = train_head(dense_init)

    sparse_init = copy.deepcopy(dense_head)
    import torch.nn.utils.prune as prune
    prune.l1_unstructured(sparse_init,name='weight',amount=0.80)
    prune.remove(sparse_init,'weight')
    sparse_mask = (sparse_init.weight != 0).float()
    sparse_head, sparse_logits, sparse_val_f1 = train_head(sparse_init, mask=sparse_mask)

    body_nonzero = sum(int((p.detach().cpu()!=0).sum()) for n,p in p80.named_parameters() if not n.startswith('head.'))
    body_total = sum(p.numel() for n,p in p80.named_parameters() if not n.startswith('head.'))
    rows=[]
    for label, head, L, vf1 in [
        ('dense_head',dense_head,dense_logits,dense_val_f1),
        ('magnitude_sparse80_head',sparse_head,sparse_logits,sparse_val_f1),
    ]:
        probs=torch.softmax(torch.tensor(L),1).numpy(); yp=probs.argmax(1)
        head_nz=int((head.weight!=0).sum()) + int((head.bias!=0).sum())
        total=body_total + sum(p.numel() for p in head.parameters())
        nonzero=body_nonzero + head_nz
        cal=calibration_summary(probs,yte).iloc[0].to_dict()
        sec,_,_=security_semantics(yte,yp,le.classes_)
        rows.append({'condition':label,'validation_macro_f1':vf1,
                     'test_macro_f1':f1_score(yte,yp,average='macro'),
                     'head_weight_sparsity':float(1-(head.weight!=0).sum().item()/head.weight.numel()),
                     'whole_model_sparsity':float(1-nonzero/total),
                     'attack_to_benign_rate':sec.loc[0,'attack_to_benign_rate'],
                     'benign_to_attack_rate':sec.loc[0,'benign_to_attack_rate'],
                     **cal})
        per_class_recall_table(yte,yp,le).to_csv(OUT_TABLE/f'{label}_per_class.csv',index=False)
    sparse_head_summary=pd.DataFrame(rows)
    sparse_head_summary.to_csv(OUT_TABLE/'dense_vs_sparse_head_refit.csv',index=False)
    display(sparse_head_summary.round(4))


## Completion

In [ ]:
print('Saved strict probes, BatchNorm recalibration, and dense-versus-sparse replacement-head controls.')
print('The strict probes use the untouched provenance-ordered test partition, with deterministic class-stratified caps recorded in the output counts.')
